In [ ]:
                                                           Programming Lab I
                                                              Handout 3
                                                   Name :  Zeeshan Akhtar
I just want to say I gained some help in some Parts from the chatgpt and gemini AI as my partner for this hangout 3 

In [4]:
import math
from itertools import combinations  
from collections import defaultdict #

# Alignment database

sequences = [
    "TSVKTYAKFVTH",
    "TSVKTYAKFSTH",
    "TSVKTYAKFVTH",
    "LSVKKYPKYVVQ"
]

#  Count amino acid frequencies (fa)

fa = defaultdict(int)

for seq in sequences:
    for aa in seq:
        fa[aa] += 1

# Total amino acids
total_fa = sum(fa.values())

print("SUM(fi) =", total_fa)
print()

#  Calculate pa

pa = {}

print("Amino Acid Frequencies")
print("-" * 40)

for aa in sorted(fa.keys()):

    pa[aa] = fa[aa] / total_fa

    print(
        f"{aa} : "
        f"fa = {fa[aa]:2d}   "
        f"pa = {pa[aa]:.4f}"
    )

# Count amino acid pairs (fab)

fab = defaultdict(int)

sequence_length = len(sequences[0])

for col in range(sequence_length):

    # Get all residues in one column
    column = [seq[col] for seq in sequences]

    # Generate unordered pairs
    for a, b in combinations(column, 2):

        pair = tuple(sorted((a, b)))

        fab[pair] += 1

# Total pairs
total_fab = sum(fab.values())

print("\nSUM(fij) =", total_fab)


#  Compute pab, eab, sab

print("\nPair Calculations")
print("-" * 75)

print(
    f"{'Pair':<6}"
    f"{'fab':>6}"
    f"{'pab':>12}"
    f"{'eab':>12}"
    f"{'sab':>8}"
)

for pair in sorted(fab.keys()):

    a, b = pair

    # Observed probability
    pab = fab[pair] / total_fab

    # Expected probability
    if a == b:
        eab = pa[a] * pa[a]
    else:
        eab = 2 * pa[a] * pa[b]

    # Score
    if pab == 0 or eab == 0:
        sab = -99
    else:
        sab = round(
            2 * math.log2(pab / eab)
        )

    print(
        f"{a+b:<6}"
        f"{fab[pair]:>6}"
        f"{pab:>12.4f}"
        f"{eab:>12.4f}"
        f"{sab:>8}"
    )

SUM(fi) = 48

Amino Acid Frequencies
----------------------------------------
A : fa =  3   pa = 0.0625
F : fa =  3   pa = 0.0625
H : fa =  3   pa = 0.0625
K : fa =  9   pa = 0.1875
L : fa =  1   pa = 0.0208
P : fa =  1   pa = 0.0208
Q : fa =  1   pa = 0.0208
S : fa =  5   pa = 0.1042
T : fa =  9   pa = 0.1875
V : fa =  8   pa = 0.1667
Y : fa =  5   pa = 0.1042

SUM(fij) = 72

Pair Calculations
---------------------------------------------------------------------------
Pair     fab         pab         eab     sab
AA         3      0.0417      0.0039       7
AP         3      0.0417      0.0026       8
FF         3      0.0417      0.0039       7
FY         3      0.0417      0.0130       3
HH         3      0.0417      0.0039       7
HQ         3      0.0417      0.0026       8
KK        12      0.1667      0.0352       4
KT         3      0.0417      0.0703      -2
LT         3      0.0417      0.0078       5
SS         6      0.0833      0.0109       6
SV         3      0.0417      0

In [ ]:
#Ex.2 (10 pts) Calculating scoring matrices
import math
from collections import defaultdict

# Standard amino acids
AMINO_ACIDS = sorted(list("ARNDCQEGHILKMFPSTWYV"))

# (a) Read alignment data
def read_alignment(filename):
    sequences = []

    with open(filename, "r") as f:
        for line in f:
            line = line.strip().replace(" ", "")
            if line:
                sequences.append(line)

    # Check same length
    length = len(sequences[0])
    for seq in sequences:
        if len(seq) != length:
            raise ValueError("All sequences must have equal length.")

    return sequences

# Count amino acid frequencies
def amino_acid_frequencies(sequences):
    counts = defaultdict(int)

    for seq in sequences:
        for aa in seq:
            counts[aa] += 1

    total = sum(counts.values())

    freq = {}
    for aa in AMINO_ACIDS:
        freq[aa] = counts[aa] / total

    return freq

# Count pair frequencies with pseudocounts
def pair_frequencies(sequences):
    pair_counts = defaultdict(lambda: 1)  # Laplace pseudocounts

    nseq = len(sequences)
    length = len(sequences[0])

    # Compare all sequence pairs
    for col in range(length):

        column = [seq[col] for seq in sequences]

        for i in range(nseq):
            for j in range(i, nseq):

                a = column[i]
                b = column[j]

                pair = tuple(sorted((a, b)))
                pair_counts[pair] += 1

    total_pairs = sum(pair_counts.values())

    pair_freq = {}
    for pair in pair_counts:
        pair_freq[pair] = pair_counts[pair] / total_pairs

    return pair_freq

# Compute log-odds scoring matrix

def compute_scores(freq, pair_freq):

    scores = {}

    for a in AMINO_ACIDS:
        for b in AMINO_ACIDS:

            pair = tuple(sorted((a, b)))

            pab = pair_freq.get(pair, 1e-10)

            # Expected probability
            if a == b:
                eab = freq[a] * freq[b]
            else:
                eab = 2 * freq[a] * freq[b]

            # Log-odds score
            score = 2 * math.log2(pab / eab)

            scores[(a, b)] = round(score)

    return scores



# Write matrix to output file
def write_matrix(scores, output_file):

    with open(output_file, "w") as out:

        # Header
        out.write("    ")
        for aa in AMINO_ACIDS:
            out.write(f"{aa:>5}")
        out.write("\n")

        # Matrix rows
        for a in AMINO_ACIDS:

            out.write(f"{a:>3} ")

            for b in AMINO_ACIDS:
                out.write(f"{scores[(a,b)]:>5}")

            out.write("\n")



# MAIN PROGRAM

# File paths for Jupyter Notebook
alignment_file = r"E:\semester #2\programming\handout-03\alignment.dat"
output_file = r"E:\semester #2\programming\handout-03\blosum_matrix.out"

# Read sequences
sequences = read_alignment(alignment_file)

# Frequencies
freq = amino_acid_frequencies(sequences)

# Pair frequencies
pair_freq = pair_frequencies(sequences)

# Compute scores
scores = compute_scores(freq, pair_freq)

# Write matrix
write_matrix(scores, output_file)

print("Scoring matrix successfully written to:")
print(output_file)

Scoring matrix successfully written to:
E:\semester #2\programming\handout-03\blosum_matrix.out


In [ ]:
import sys

# Read FASTA file
def read_fasta(filename):

    sequence = ""

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                continue

            sequence += line

    return sequence


# Read scoring matrix
def read_matrix(filename):

    matrix = {}

    with open(filename, "r") as f:

        lines = [line.strip() for line in f if line.strip()]

    headers = lines[0].split()

    for line in lines[1:]:

        parts = line.split()

        row = parts[0]

        matrix[row] = {}

        for aa, value in zip(headers, parts[1:]):

            matrix[row][aa] = int(value)

    return matrix


# Needleman-Wunsch Algorithm
def needleman_wunsch(seq1, seq2, matrix, gap):

    n = len(seq1)
    m = len(seq2)

    dp = [[0]*(m+1) for _ in range(n+1)]
    trace = [[None]*(m+1) for _ in range(n+1)]

    # Initialization
    for i in range(1, n+1):

        dp[i][0] = dp[i-1][0] + gap
        trace[i][0] = "U"

    for j in range(1, m+1):

        dp[0][j] = dp[0][j-1] + gap
        trace[0][j] = "L"

    # Fill matrix
    for i in range(1, n+1):

        for j in range(1, m+1):

            match = dp[i-1][j-1] + matrix[seq1[i-1]][seq2[j-1]]

            delete = dp[i-1][j] + gap

            insert = dp[i][j-1] + gap

            best = max(match, delete, insert)

            dp[i][j] = best

            if best == match:
                trace[i][j] = "D"

            elif best == delete:
                trace[i][j] = "U"

            else:
                trace[i][j] = "L"

    # Traceback
    aligned1 = []
    aligned2 = []

    i = n
    j = m

    while i > 0 or j > 0:

        direction = trace[i][j]

        if direction == "D":

            aligned1.append(seq1[i-1])
            aligned2.append(seq2[j-1])

            i -= 1
            j -= 1

        elif direction == "U":

            aligned1.append(seq1[i-1])
            aligned2.append("-")

            i -= 1

        else:

            aligned1.append("-")
            aligned2.append(seq2[j-1])

            j -= 1

    aligned1.reverse()
    aligned2.reverse()

    return dp[n][m], "".join(aligned1), "".join(aligned2)

# Similarity line

def similarity_line(a1, a2, matrix):

    line = ""

    for x, y in zip(a1, a2):

        if x == y:

            line += "|"

        elif x != "-" and y != "-" and matrix[x][y] > 0:

            line += ":"

        else:

            line += " "

    return line


# Print alignment
def print_alignment(a1, a2, matrix, width=80):

    middle = similarity_line(a1, a2, matrix)

    for i in range(0, len(a1), width):

        print(a1[i:i+width])
        print(middle[i:i+width])
        print(a2[i:i+width])
        print()


# MAIN
def main():

    if len(sys.argv) != 5:

        print("No command-line arguments detected.")
        print("Using default values...\n")

        gap = -5

        matrix_file = r"E:\semester #2\programming\handout-03\blosum62.dat"

        seq1 = "THRQATWQPPLERMANGRQVE"

        seq2 = "RAYMQNDLVKVRYYACHT"

    else:

        gap = int(sys.argv[1])

        matrix_file = sys.argv[2]

        seq1 = read_fasta(sys.argv[3])

        seq2 = read_fasta(sys.argv[4])

    # Read matrix
    matrix = read_matrix(matrix_file)

    # Run alignment
    score, a1, a2 = needleman_wunsch(seq1, seq2, matrix, gap)

    # Output
    print("Alignment Score:", score)
    print()

    print_alignment(a1, a2, matrix)


if __name__ == "__main__":

    main()

No command-line arguments detected.
Using default values...

Alignment Score: -16

THRQATWQPPLERMANGRQVE
  | |  |  | ::       
--R-AYMQNDLVKVRYYACHT



In [ ]:
### Ex.3 RNAS1 --- Three-way comparison
# for Horse vs Whale
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum50.txt")

seq1 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_horse.fasta")
seq2 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_minke-whale.fasta")

score, a1, a2 = needleman_wunsch(seq1, seq2, matrix, -8)

print("Alignment Score:", score)
print()

print_alignment(a1, a2, matrix)

Alignment Score: 683

KESPAMKFERQHMDSGSTSSSNPTYCNQMMKRRNMTQGWCKPVNTFVHEPLADVQAICLQKNITCKNGQSNCYQSSSSMH
:|||||||:|||||||::  :|| |||||| || |||| |||||||||| | ||:|:| |||: ||||::|||:|:|:||
RESPAMKFQRQHMDSGNSPGNNPNYCNQMMMRRKMTQGRCKPVNTFVHESLEDVKAVCSQKNVLCKNGRTNCYESNSTMH

ITDCRLTSGSKYPNCAYQTSQKERHIIVACEGNPYVPVHFDASVEVST
||||| |  ||||||||:|||||:||||||||||||||||| |  |  
ITDCRQTGSSKYPNCAYKTSQKEKHIIVACEGNPYVPVHFDNS--V--



In [33]:
#Horse vs Kangaro0
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum50.txt")

seq1 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_horse.fasta")
seq2 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_red-kangaroo.fasta")

score, a1, a2 = needleman_wunsch(seq1, seq2, matrix, -8)

print("Alignment Score:", score)
print()

print_alignment(a1, a2, matrix)

Alignment Score: 521

KESPAMKFERQHMDSGSTSSSNPTYCNQMMKRRNMTQGWCKPVNTFVHEPLADVQAICLQKNITCKNGQSNCYQSSSSMH
 |:|| ||:|||||:  :::|:  ||| ||| |:|| | |||:|||:||| : | |:| |:|:|||||::|||:|:| : 
-ETPAEKFQRQHMDTEHSTASSSNYCNLMMKARDMTSGRCKPLNTFIHEPKSVVDAVCHQENVTCKNGRTNCYKSNSRLS

ITDCRLTSGSKYPNCAYQTSQKERHIIVACEGNPYVPVHFDASVEVST
||:|| |  |||||| |:||   ::|||||||  ||||||||   |  
ITNCRQTGASKYPNCQYETSNLNKQIIVACEGQ-YVPVHFDA-Y-V--



In [34]:
#Whale vs Kangaroo
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum50.txt")

seq1 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_minke-whale.fasta")
seq2 = read_fasta(r"E:\semester #2\programming\handout-03\RNAS1_red-kangaroo.fasta")

score, a1, a2 = needleman_wunsch(seq1, seq2, matrix, -8)

print("Alignment Score:", score)
print()

print_alignment(a1, a2, matrix)

Alignment Score: 572

RESPAMKFQRQHMDSGNSPGNNPNYCNQMMMRRKMTQGRCKPVNTFVHESLEDVKAVCSQKNVLCKNGRTNCYESNSTMH
 |:|| ||||||||: :|  :: |||| ||  | || |||||:|||:||    | ||| |:|| |||||||||:||| : 
-ETPAEKFQRQHMDTEHSTASSSNYCNLMMKARDMTSGRCKPLNTFIHEPKSVVDAVCHQENVTCKNGRTNCYKSNSRLS

ITDCRQTGSSKYPNCAYKTSQKEKHIIVACEGNPYVPVHFDNSV
||:|||||:|||||| |:||   |:|||||||  |||||||  |
ITNCRQTGASKYPNCQYETSNLNKQIIVACEGQ-YVPVHFDAYV



In [35]:
### Ex.3 Global vs. local alignment
#blosum62.txt halodurans.fasta lentus.fasta
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum62.txt")

seq1 = read_fasta(r"E:\semester #2\programming\handout-03\halodurans.fasta")
seq2 = read_fasta(r"E:\semester #2\programming\handout-03\lentus.fasta")

score, a1, a2 = needleman_wunsch(seq1, seq2, matrix, -8)

print("GLOBAL ALIGNMENT SCORE:", score)
print()
print_alignment(a1, a2, matrix)

score, identity, similarity, a1, a2 = smith_waterman(
    seq1,
    seq2,
    matrix,
    -8
)

print("LOCAL ALIGNMENT SCORE:", score)
print("IDENTITY (%):", identity)
print("SIMILARITY (%):", similarity)
print()
print(a1)
print(a2)


GLOBAL ALIGNMENT SCORE: 187

MRQSLKVMVLSTVALLFMANPAAASEEKKEYLIVVEPEEVSAQSVEESYDVDVIHEFEEIPVIHAELTKKELKKLKKDPN
  |       | |       |         :             :  |                            :   
-AQ-------S-V-------P---------W------------GI--S----------------------------R---

VKAIEKNAEVTISQTVPWGISFINTQQAHNRGIFGNGARVAVLDTGIASHPDLRIAGGASFISSEPSYHDNNGHGTHVAG
|   :  |        |   :      |||||: |:| :||||||||::|||| | |||||:  |||  | |||||||||
V---Q--A--------P---A------AHNRGLTGSGVKVAVLDTGISTHPDLNIRGGASFVPGEPSTQDGNGHGTHVAG

TIAALNNSIGVLGVAPSADLYAVKVLDRNGSGSLASVAQGIEWAINNNMHIINMSLGSTSGSSTLELAVNRANNAGILLV
||||||||||||||||||:|||||||  :||||::|:|||:||| || ||: |:|||| | |:||| ||| | : |:|:|
TIAALNNSIGVLGVAPSAELYAVKVLGASGSGSVSSIAQGLEWAGNNGMHVANLSLGSPSPSATLEQAVNSATSRGVLVV

GAAGNTGRQGVNYPARYSGVMAVAAVDQNGQRASFSTYGPEIEISAPGVNVNSTYTGNRYVSLSGTSMATPHVAGVAALV
 |:||:|   ::|||||:  ||| | |||  ||||| ||  ::| |||||| ||| |: | ||:||||||||||| ||||
AASGNSGAGSISYPARYANAMAVGATDQNNNRASFSQYGAGLDIVAPGVNVQSTYPGSTYASLNGTSMATPHVAGAA

In [38]:
#waterman.py -8 blosum62.txt halodurans.fasta lentus.fasta
seq1 = read_fasta(r"E:\semester #2\programming\handout-03\halodurans.fasta")
seq2 = read_fasta(r"E:\semester #2\programming\handout-03\lentus.fasta")
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum62.txt")

score, identity, similarity, a1, a2 = smith_waterman(
    seq1,
    seq2,
    matrix,
    -8
)

print("LOCAL ALIGNMENT SCORE:", score)
print("IDENTITY (%):", identity)
print("SIMILARITY (%):", similarity)
print()
print(a1)
print(a2)

LOCAL ALIGNMENT SCORE: 916
IDENTITY (%): 65.4275092936803
SIMILARITY (%): 79.5539033457249

SQTVPWGISFINTQQAHNRGIFGNGARVAVLDTGIASHPDLRIAGGASFISSEPSYHDNNGHGTHVAGTIAALNNSIGVLGVAPSADLYAVKVLDRNGSGSLASVAQGIEWAINNNMHIINMSLGSTSGSSTLELAVNRANNAGILLVGAAGNTGRQGVNYPARYSGVMAVAAVDQNGQRASFSTYGPEIEISAPGVNVNSTYTGNRYVSLSGTSMATPHVAGVAALVKSRYPSYTNNQIRQRINQTATYLGSPSLYGNGLVHAGRATQ
AQSVPWGISRVQAPAAHNRGLTGSGVKVAVLDTGISTHPDLNIRGGASFVPGEPSTQDGNGHGTHVAGTIAALNNSIGVLGVAPSAELYAVKVLGASGSGSVSSIAQGLEWAGNNGMHVANLSLGSPSPSATLEQAVNSATSRGVLVVAASGNSGAGSISYPARYANAMAVGATDQNNNRASFSQYGAGLDIVAPGVNVQSTYPGSTYASLNGTSMATPHVAGAAALVKQKNPSWSNVQIRNHLKNTATSLGSTNLYGSGLVNAEAATR


In [ ]:
#Ex.4  Affine gaps
#(a)  Modify the global sequence alignment implementation 

def read_matrix(filename):

    matrix = {}

    with open(filename, "r") as f:

        lines = []

        for line in f:

            line = line.strip()

            # Skip comments or empty lines
            if line == "" or line.startswith("#"):
                continue

            lines.append(line)

    # Header row
    headers = lines[0].split()

    # Read scores
    for line in lines[1:]:

        parts = line.split()

        row_aa = parts[0]

        scores = parts[1:]

        for col_aa, score in zip(headers, scores):

            matrix[(row_aa, col_aa)] = int(score)

    return matrix

In [ ]:
#(B)Testing. Perform the sequence alignment 
matrix = read_matrix(r"E:\semester #2\programming\handout-03\blosum62.txt")

seq1 = read_fasta(r"E:\semester #2\programming\handout-03\GLB7A_CHITH.fasta")

seq2 = read_fasta(r"E:\semester #2\programming\handout-03\GLBE_CHITH.fasta")

score, a1, a2 = needleman_wunsch_affine(
    seq1,
    seq2,
    matrix,
    -8,
    -2
)

print("GLOBAL ALIGNMENT SCORE:", score)
print()

print(a1)
print(a2)

GLOBAL ALIGNMENT SCORE: 358

MKFFAVLALCIVGAIASPLSADQAALVKSTWAQVRNSEVEILAAVFTAYPDIQARFPQFAGKDVASIKDTGAFATHAGRIVGFVSEIIALIGNESNAPAVQTLVGQLAASHKARGISQAQFNEFRAGLVSYVSSNVAWNAAAESAWTAGLDNIFGLLFAAL
MKFI-ILALCV--AAASALSGDQIGLVQSTYGKVKGDSVGILYAVFKADPTIQAAFPQFVGKDLDAIKGGAEFSTHAGRIVGFLGGVI------DDLPNIGKHVDALVATHKPRGVTHAQFNNFRAAFIAYLKGHVDYTAAVEAAWGATFDAFFGAVFAKM
